In [1]:
import json
import os
import re
import requests
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from google_auth_oauthlib.flow import InstalledAppFlow
from openai import OpenAI

load_dotenv()

CLIENT_ID = os.getenv("GOOGLE_API_OAUTH_CLIENT_ID")
CLIENT_SECRET = os.getenv("GOOGLE_API_OAUTH_CLIENT_SECRET")
client_config = {
    "installed": {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
    }
}

ACCOUNT_ID = "114674571764534564133"
LOCATION_ID = "8962787873732607458"
SCOPES = ["https://www.googleapis.com/auth/business.manage"]

openai_client = OpenAI(api_key=os.getenv("CONFIG__OPENAI__KEY"))

In [2]:
# 1. Authenticate and get your token
flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
credentials = flow.run_local_server(port=0)

headers = {
    "Authorization": f"Bearer {credentials.token}"
}

# 2. Get your Account ID
print("Fetching Accounts...")
accounts_url = "https://mybusinessaccountmanagement.googleapis.com/v1/accounts"
accounts_response = requests.get(accounts_url, headers=headers).json()

print(json.dumps(accounts_response, indent=2))

# 3. Get your Location ID(s)
# Grab the first account name (it will look like "accounts/114674571764534564133")
if "accounts" in accounts_response:
    account_name = accounts_response["accounts"][0]["name"]
    print(f"\nUsing Account: {account_name}")
    
    # We use a readMask to specify we only want the name and title of the location
    locations_url = f"https://mybusinessbusinessinformation.googleapis.com/v1/{account_name}/locations?readMask=name,title"
    locations_response = requests.get(locations_url, headers=headers).json()
    
    print("\nFetching Locations...")
    print(json.dumps(locations_response, indent=2))
else:
    print("No accounts found for this authenticated user.")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=882398647440-shei634mkfbeujau8inuv4imeuv9n1dq.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A59590%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbusiness.manage&state=hJ5go9UyDFwcrl1cT1uQkQuwkJVN50&code_challenge=zUz2JUNs1ZTKt55cE3C57rLxf9rITiS6xo2bM_5dCyE&code_challenge_method=S256&access_type=offline
Fetching Accounts...
{
  "accounts": [
    {
      "name": "accounts/114674571764534564133",
      "accountName": "\u0141ukasz Regulski",
      "type": "PERSONAL",
      "verificationState": "UNVERIFIED",
      "vettedState": "NOT_VETTED"
    }
  ]
}

Using Account: accounts/114674571764534564133

Fetching Locations...
{
  "locations": [
    {
      "name": "locations/8962787873732607458",
      "title": "Pensjonat \"Rybical 66\""
    }
  ]
}


In [8]:
def get_rybical_reviews():
    flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
    creds = flow.run_local_server(port=0)
    headers = {
        "Authorization": f"Bearer {creds.token}",
        "Content-Type": "application/json",
    }
    url = f"https://mybusiness.googleapis.com/v4/accounts/{ACCOUNT_ID}/locations/{LOCATION_ID}/reviews"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        reviews = data.get("reviews", [])
        print(f"Fetched {len(reviews)} reviews.")
        return reviews
    print(f"Error {response.status_code}: {response.text}")
    return None


def analyze_review(comment: str) -> dict:
    """Extract good/bad points from review text. Returns dict with good_points, bad_points."""
    if not comment or str(comment).strip() == "":
        return {"good_points": "", "bad_points": ""}

    prompt = """You are analyzing a hotel guest review. Extract briefly what the guest pointed as GOOD and as BAD.
Reply with a single JSON object only, no markdown, no code block, with two keys: "good_points" and "bad_points".
Each value: short bullet list or comma-separated items; if nothing for one side, use empty string "".
Review text:
"""
    
    resp = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt + comment[:4000]}],
        temperature=0,
    )
    text = resp.choices[0].message.content.strip()
    if text.startswith("```"):
        text = re.sub(r"^```json\s*|```$", "", text, flags=re.MULTILINE)
    return json.loads(text)

def review_to_row(r):
    comment = r.get("comment") or ""
    create_time = r.get("createTime") or ""
    try:
        dt = datetime.fromisoformat(create_time.replace("Z", "+00:00"))
        date_str = dt.strftime("%Y-%m-%d")
    except Exception:
        date_str = create_time
    rating = r.get("starRating", "")
    reviewer = (r.get("reviewer") or {}).get("displayName", "")
    reply_obj = r.get("reviewReply") or {}
    our_response = reply_obj.get("comment", "") if reply_obj else ""
    analysis = analyze_review(comment)
    return {
        "date": date_str,
        "reviewer": reviewer,
        "rating": rating,
        "comment": comment,
        "good_points": analysis.get("good_points", ""),
        "bad_points": analysis.get("bad_points", ""),
        "our_response": our_response,
    }

In [9]:
my_reviews = get_rybical_reviews()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=882398647440-shei634mkfbeujau8inuv4imeuv9n1dq.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49244%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbusiness.manage&state=mMplN2o1bH85NZGFBQoQBbSNN1l6lo&code_challenge=Yybu0F4dS8dB3oC34G63t_DU72hCbq6cdXZW0qvw3u8&code_challenge_method=S256&access_type=offline
Fetched 50 reviews.


In [10]:
rows = [review_to_row(r) for r in (my_reviews or [])]
reviews_df = pd.DataFrame(rows)

In [11]:
reviews_df

,date,reviewer,rating,comment,good_points,bad_points,our_response
0,2025-11-25,Piotr Ostapczuk,FIVE,(Translated by Google) I stayed with my wife a...,"place to relax by the water, sauna, good cuisi...",cleanliness of the bathroom,
1,2025-10-12,Grzegorz Sujkowski,FIVE,(Translated by Google) Thanks to the owners an...,"peace, quiet, comfort, friendly atmosphere, de...",,
2,2025-09-21,Zbyszek M,FIVE,,,,"Dziękujemy za miłą opinię, zapraszamy ponownie."
3,2025-10-01,Aneta Biela,FIVE,(Translated by Google) Rybical 66 Guesthouse i...,"oasis of peace, marina with kayaks and pedal b...",,
4,2025-09-21,Golden Eye,FOUR,,,,
5,2025-08-31,Marta Szczupak,FIVE,"(Translated by Google) A beautifully situated,...","beautiful location, intimate guesthouse, atten...",,Dziękujemy za miłe słowa. Mamy nadzieję gościć...
6,2025-08-24,Dorota Ferenc,FIVE,(Translated by Google) We wholeheartedly recom...,"beautiful location, spacious grounds, comforta...",lake not in a quiet zone,Pięknie dziękujemy za miła opinię. Zapraszamy ...
7,2025-08-24,Agnieszka Drwal,FIVE,"(Translated by Google) A wonderful place, wond...","wonderful place, wonderful owners, delicious food",,Pięknie dziękujemy za miła opinię. Zapraszamy ...
8,2025-08-20,Marta Andrearczyk,FIVE,(Translated by Google) A comfortable apartment...,"comfortable apartment, beautiful location, rig...",monotonous breakfasts,Pięknie dziękujemy za miła opinię. Zapraszamy ...
9,2025-08-18,Robert Dworak,FIVE,"(Translated by Google) Fantastic place, I high...","fantastic place, highly recommend",,Pięknie dziękujemy za miła opinię. Zapraszamy ...
